# 01 – Characters

Esplorazione e data cleaning del dataset `characters.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `character_mal_id` | ID univoco del personaggio su MyAnimeList |
| `url` | URL della pagina MAL del personaggio |
| `name` | Nome del personaggio (in inglese) |
| `name_kanji` | Nome in giapponese |
| `image` | URL dell'immagine del personaggio |
| `favorites` | Numero di utenti che hanno aggiunto il personaggio ai preferiti |
| `about` | Descrizione testuale del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_characters = pd.read_csv('../datasets/characters.csv')
print(f'Shape: {df_characters.shape}')
df_characters.info()
df_characters.head()

Il dataset contiene **209.963** righe che corispondono ai personaggi e **7** colonne che corrispondo agli attributi per ciascuno. 

Notiamo che il numero di righe non nulle non corrisponde al numero totale di righe, il che significa che ci sono dei valori mancanti da gestire. 

Notiamo anche che `character_mal_id` e `favorite` sono di tipo `float64` invece di `int` il che potrebbe portare a errori, occupa più memoria del necessario e per numeri molto grandi potrebbe ridurre la leggibilità. 

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [61]:
n_originale = len(df_characters) #calcoliamo il numero totale di righe

mask_dup = df_characters.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_characters[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_characters.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_characters):,}')

Righe totali coinvolte in duplicazioni : 796
  → prime occorrenze mantenute         : 340
  → occorrenze extra rimosse           : 456

Righe prima della rimozione : 209,963
Righe dopo la rimozione     : 209,507


Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`
 Questa è la chiave primaria del dataset: ogni riga deve avere un ID univoco e non nullo.

In [62]:
analyze(df_characters['character_mal_id'])


  Nome serie                     character_mal_id
  dtype                          float64
  Dimensione                     209,507
  Range indice                   0 … 209962
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,507
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     1  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       209,506   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   209,506  (100.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            1.0000
  Max                            282,284.00
  Range                          282,283.00
  Media                          150,380.10
  Mediana                        159,933.50
  Moda/e                   

**Osservazioni:**
- La colonna è `float64` invece di `int`, pur contenendo solo ID interi. Questo è un problema perché occupa più memoria del necessario e può ridurre la leggibilità per numeri grandi.  
- Su 209.507 righe, c'è esattamente 1 valore null. Essendo la chiave primaria, non può essere tollerato e quindi quella riga va rimossa.  
- Tutti i 209.506 valori non nulli sono unici (100% unicità), coerente con il ruolo di chiave primaria. I duplicati sono già stati eliminati in precedenza.  
- Il range di valori è compreso tra Min: 1 e Max: 282.284. Ma il dataset ha solo 209.506 righe, quindi ci sono gap negli ID. 
- La media (150.380) e la mediana (159.933) sono vicine, e la distribuzione per fasce mostra percentuali abbastanza costanti (~5-6% per fascia). Gli ID sono distribuiti in modo relativamente uniforme sull'intero range.   
- Non ci sono outliers. Con IQR = 150.688, le soglie outlier sarebbero -151.636 e 451.116. I nostri ID vanno da 1 a 282.284. Entrambi rientrano dentro le soglie.

**Pulizie necessarie:**
- Convertiamo i valori di tipo `float64` a `int`.
- Rimuoviamo le righe con `character_mal_id` null in quanto è chiave primaria. 
- Non ci sono duplicati da rimuovere in quanto sono già stati gestiti durante la   
  rimozione dei duplicati esatti.

In [63]:
df_characters.dropna(subset=['character_mal_id'], inplace=True)
df_characters['character_mal_id'] = df_characters['character_mal_id'].astype(int)
print(f'Righe dopo pulizia character_mal_id: {len(df_characters):,}')

Righe dopo pulizia character_mal_id: 209,506


### 2.2 `url`
Questa colonna contiene il link alla pagina del personaggio su MyAnimeList.

In [64]:
analyze(df_characters['url'])

pattern = r'^https://myanimelist\.net/character/\d+/\S+$'                                                               
non_conformi = df_characters[~df_characters['url'].str.match(pattern)]                                                  
print(f'URL non conformi al formato atteso: {len(non_conformi)}') 
print(non_conformi[['character_mal_id', 'url', 'name']].to_string(index=True))


  Nome serie                     url
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
  Stringhe vuote                 0
  Valori unici                   209,506  (100.00%)
  Valori duplicati               0

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  39  caratteri
  Lunghezza max                  94  caratteri
  Lunghezza media                51.19  caratteri
  Lunghezza mediana              52.0000  caratteri
  Dev. standard lunghezza        4.59
  IQR lunghezza                  7.0000

Valori di lunghezza estrema
─────────────────


**Osservazioni**
- Non ci sono dei valori nulli o duplicati. 
- Tutti i 209.506 URL sono unici (100%). 
- Lunghezza varia tra 39 e 94 caratteri. La variazione è interamente dovuta alla lunghezza del nome in coda all'URL.
- Dall' analisi del pattern risulta che una delle righe ha URL non conforme al formato atteso. Essendo solo una, abbiamo verificato manualmente nel file CSV e abbiamo determinato che l'errore è dovuto a uno spazio vuoto alla fine dell URL. 

**Pulizia necessaria.**  
I null e i duplicati sono già stati rimossi durante la pulizia di `character_mal_id`. L'unica pulizia da fare sarebbe rimuobere lo spazio vuoto in fondo al URL non conforme al format.

In [65]:
df_characters['url'] = df_characters['url'].str.strip()
print(f'Null in url       : {df_characters["url"].isna().sum()}')
print(f'Duplicati in url  : {df_characters["url"].duplicated().sum()}')

Null in url       : 0
Duplicati in url  : 0


### 2.3 `name`

Nome del personaggio in lingua originale. A differenza di `character_mal_id`, i duplicati sono attesi in quanto personaggi diversi possono condividere lo stesso nome.
Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa, come già fatto per `url`.

In [66]:
analyze(df_characters['name'])
df_characters['name'] = df_characters['name'].str.strip()


  Nome serie                     name
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
  Stringhe vuote                 0
  Valori unici                   162,787  (77.70%)
  Valori duplicati               46,719

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  54  caratteri
  Lunghezza media                10.58  caratteri
  Lunghezza mediana              11.0000  caratteri
  Dev. standard lunghezza        4.60
  IQR lunghezza                  8.0000

Valori di lunghezza estrema
─────────────

**Osservazioni**
- Non ci sono stringhe null o vuote. Sono già state gestite con la pulizia di `character_mal_id`.
- Ci sono 46.719 duplicati (22.3%) che sono attesi e non problematici.
- Ci sono solo 162.787 nomi unici su 209.506 che conferma che i nomi non sono identificatori univoci.     
- Lunghezza minima è di 1 carattere. Dalla ricerca notiamo che esistono personaggi con nome a singola lettera. Quindi questo non rappresenta un errore.
- Lunghezza massima è di 54 caratteri che corrisponde al personaggio Éléonore Albertine Le Blanc de La Blois de La Vallière. Dalla ricerca risulta che il personaggio esiste e il nome è corretto.        

**Nessuna pulizia necessaria.**  

### 2.4 `name_kanji`
Nome del personaggio in caratteri giapponesi. Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa.  

In [67]:
analyze(df_characters['name_kanji'])
df_characters['name_kanji'] = df_characters['name_kanji'].str.strip()


  Nome serie                     name_kanji
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               154,138  (73.57%)
  Null / NaN                     55,368  (26.43%)
  Stringhe vuote                 0
  Valori unici                   129,166  (61.65%)
  Valori duplicati               24,972

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  47  caratteri
  Lunghezza media                5.16  caratteri
  Lunghezza mediana              5.0000  caratteri
  Dev. standard lunghezza        2.82
  IQR lunghezza                  3.0000

Valori di lunghezza estrema
────

**Osservazioni**
- Ci sono 55.368 null (26.4%). Questo è un valore atteso che si può riferire a personaggi occidentali o di origine non giapponese che non hanno un nome kanji.
- Ci sono 24.972 duplicati che come per name, sono attesi.
- Lunghezza minima è di 1 carattere che come per `name`, è valido.              
- Lunghezza massima è di 47 caratteri. Ci sono personaggi con alias multipli. 

**Nessuna pulizia necessaria.**  

### 2.5 `image`
URL dell'immagine del personaggio su MyAnimeList.  

In [68]:
analyze(df_characters['image'])

pattern_img = r'^https://cdn\.myanimelist\.net/(images/characters/\d+/\d+\.jpg|img/sp/icon/apple-touch-icon-256\.png)$' 
non_conformi_img = df_characters[~df_characters['image'].str.match(pattern_img)]                                        
print(f'Immagini non conformi al formato atteso: {len(non_conformi_img)}')                                              
if len(non_conformi_img) > 0:                                                                                           
    print(non_conformi_img[['character_mal_id', 'image', 'name']].to_string(index=True))  


  Nome serie                     image
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
  Stringhe vuote                 0
  Valori unici                   186,327  (88.94%)
  Valori duplicati               23,179

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  55  caratteri
  Lunghezza max                  64  caratteri
  Lunghezza media                58.99  caratteri
  Lunghezza mediana              58.0000  caratteri
  Dev. standard lunghezza        1.85
  IQR lunghezza                  1.0000

Valori di lunghezza estrema
───────────

**Osservazione** 
- Non ci sono stringhe vuote o null. Tutta la colonna è popolata. 
- Ci sono 23.179 URL duplicati (11.1%). Tutti corrispondono a un link che porta a un'immagine placeholder. A ogni personaggio senza immagine reale viene assegnato il logo MAL. Non è un errore e non è necessaria nessuna pulizia. 
- Ci sono 186.327 URL unici (88.94%). Tutti i personaggi con immagine reale hanno un URL univoco.
- Lunghezza stringhe nel range 55–64 caratteri, nessun outlier.
- Tutte le URL seguono il formato corretto senza nessuna anomalia strutturale.
- Nella distribuzione delle lunghezze, le fasce 59–63 hanno 0 occorrenze, il che significa che le URL si dividono in due gruppi: percorso /images/characters/ (55–58 caratteri) e percorso placeholder /img/sp/icon/ (64 caratteri). 


**Nessuna pulizia necessaria**


### 2.6 `favorites`

Numero di utenti che hanno aggiunto il personaggio ai propri preferiti. È una metrica di popolarità.

In [69]:
analyze(df_characters['favorites'])


  Nome serie                     favorites
  dtype                          float64
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               209,506  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           131,681  (62.85%)
  Positivi                       77,825   (37.15%)
  Negativi                       0   (0.00%)
  Valori unici                   2,253  (1.08%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0.0000
  Max                            175,632.00
  Range                          175,632.00
  Media                          57.6972
  Mediana                        0.0000
  Moda/e                         0.0000


**Osservazioni**
- Nessun null, nessun valore negativo. La colonna è completamente popolata e non contiene errori. 
- Il `dtype` è `float64` invece di `int64`, come già visto per `character_mal_id`. 
- Il 62.85% dei valori è 0 (131.681 personaggi). Questo è un valore atteso. Significa che la grande maggioranza dei personaggi non è stata aggiunta nel preferito come per esemio personaggi di serie nuove.
- Si nota una distribuzione asimetrica con Mediana = 0, Media = 57.7 e Coefficiente di Variazione = 2077%. Significa che pochi personaggi vengo aggiunti di più nei preferiti. 
- P95 = 47 significa che il 95% dei personaggi ha ≤ 47 preferiti.
- Si notano 30,242 outlier secondo IQR. Non sono dati sporchi ma semplicemente il metodo IQR non è adatto a questa colonna. Qualsiasi valore maggiore di 5 viene classificato outlier ma avere più di 5 preferiti non è un anomalia. 
- Ci sono 2.253 valori unici su 209.506 righe (1.08%). Non è un errore in quanto molti personaggi condividono gli stessi conteggi bassi.

**Pulizie necessarie:**
- Convertire il `dtype` float64 a `int`

In [70]:
df_characters['favorites'] = df_characters['favorites'].astype(int)
print(f'favorites dtype   : {df_characters["favorites"].dtype}')

favorites dtype   : int64


### 2.7 `about`
Testo libero con la descrizione del personaggio.

In [71]:
analyze(df_characters['about'])

dup_mask = df_characters['about'].duplicated(keep=False) & df_characters['about'].notna()
dup_texts = df_characters.loc[dup_mask, 'about'].value_counts()
print(f'Testi duplicati distinti: {len(dup_texts)}')
print(dup_texts.head(20))


  Nome serie                     about
  dtype                          str
  Dimensione                     209,506
  Range indice                   0 … 209962
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   209,506
  Valori non nulli               112,736  (53.81%)
  Null / NaN                     96,770  (46.19%)
  Stringhe vuote                 0
  Valori unici                   109,515  (52.27%)
  Valori duplicati               3,221

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  3  caratteri
  Lunghezza max                  14,449  caratteri
  Lunghezza media                393.66  caratteri
  Lunghezza mediana              277.0000  caratteri
  Dev. standard lunghezza        440.95
  IQR lunghezza                  317.0000

Valori di lunghezza estrem

**Osservazioni**
- Notiamo che 46.19% è null (96.770). Questo risultato è atteso in quanto i personaggi secondari o poco conosciuti possono non avere una descrizione.
- Ci sono 3.221 duplicati. Da questi, la frase "No voice actors have been added to this character. Help improve our database…" compare 199 volte. Non abbiamo informazioni precise sul resto dei duplicati. Abbiamo indagato più in dettaglio e sembra che il resto dei duplicati è composto dalla frase "Appears in episode X.". Non sono dati sporchi ma contenuto boilerplate.
- La stringa più corta è "ANN" (3 caratteri) che può essere un dato spazzatura o un tag residuo. Da verificare manualmente.
- Il 87.7% dei valori non-null ha lunghezza ≤ 725 caratteri. La maggior parte delle descrizioni è breve. Solo il 10.6% supera i 725 caratteri.
- Risluta 1 URL nel testo e 120 valori con @. Questo può essere rumore minore, probabilmente link o menzioni.
- Lingua prevalentemente inglese (parole più comuni: the, to, and, a, of…), ma i 2.314 caratteri unici indicano presenza di testo non-latino (giapponese, arabo, ecc.).

**Nessuna pulizia necessaria.**

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [72]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_characters):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_characters):>10,}')
print()
print('Dtypes finali:')
print(df_characters.dtypes)
print()
print('Valori mancanti residui:')
print(df_characters.isnull().sum())
print()
df_characters.to_csv('../datasets_cleaned/characters_clean.csv', index=False)
print('Salvato: datasets_cleaned/characters_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    209,963
Righe dopo cleaning  :    209,506
Righe rimosse totali :        457

Dtypes finali:
character_mal_id    int64
url                   str
name                  str
name_kanji            str
image                 str
favorites           int64
about                 str
dtype: object

Valori mancanti residui:
character_mal_id        0
url                     0
name                    0
name_kanji          55368
image                   0
favorites               0
about               96770
dtype: int64

Salvato: datasets_cleaned/characters_clean.csv
